# IT594 Project: Notebook 05, mechanism ablation and where the hypothesis fails

**Part A, controlled ablation.** The systems never see `product_features`, where color is stored. We add **only the color**
(`Color: navy.`) to each product's text, change nothing else, and rerun all three systems. If dense's violations fall sharply,
dense violates color mainly because the color is not in the text it reads, not because of the model itself.

**Part B, where the hypothesis fails.** Per color query: is dense worse, equal or better than BM25 on violation, and what do the
queries where it is *not* worse have in common?

Relevance and violation are reported in separate tables, as always.

## 1. Setup and load data

In [1]:
import os, re, sys
import numpy as np, pandas as pd

sys.path.insert(0, "../src")
from metrics import ndcg_at_k, constraint_violation_rate, paired_bootstrap
from color_checker import query_colors, color_check
from retrieval import rrf

PATH = "../data/WANDS/dataset/"
products = pd.read_csv(PATH + "product.csv", sep="\t")
queries  = pd.read_csv(PATH + "query.csv",   sep="\t")
labels   = pd.read_csv(PATH + "label.csv",   sep="\t")

GAIN = {"Exact": 2, "Partial": 1, "Irrelevant": 0}
labels["gain"] = labels["label"].map(GAIN)
qrels = {q: dict(zip(g["product_id"], g["gain"])) for q, g in labels.groupby("query_id")}
qtext = dict(zip(queries["query_id"], queries["query"]))
qclass = dict(zip(queries["query_id"], queries["query_class"]))
all_q = list(queries["query_id"])

def parse_features(s):
    d = {}
    if pd.isna(s):
        return d
    for pair in s.split("|"):
        if ":" in pair:
            k, v = pair.split(":", 1)
            d[k.strip().lower()] = v.strip()
    return d

feat = products["product_features"].apply(parse_features)
products["pcolor"] = feat.apply(lambda d: d.get("color") or d.get("primarycolor") or "")
pcolor = dict(zip(products["product_id"], products["pcolor"]))

for c in ["product_name", "product_class", "product_description"]:
    products[c] = products[c].fillna("")
pname = dict(zip(products["product_id"], products["product_name"]))
pids = products["product_id"].values
print(products.shape, queries.shape, labels.shape)
print(f"products with a color value: {(products['pcolor'] != '').mean():.1%}")

(42994, 10) (480, 3) (233448, 5)
products with a color value: 38.7%


## 2. Two versions of the product text

- **base:** name + class + first 500 characters of description (as in every earlier notebook)
- **+color:** the same, plus `Color: <value>.` when the product has a color value. Nothing else changes.

In [2]:
base_text = products["product_name"] + ". " + products["product_class"] + ". " + products["product_description"].str.slice(0, 500)
color_text = base_text + np.where(products["pcolor"] != "", " Color: " + products["pcolor"] + ".", "")
TEXTS = {"base": base_text, "+color": color_text}
print(color_text[products["pcolor"] != ""].iloc[0][-120:])

any bedroom , acting both as a decorative piece as well as a place to give comfort after a hard day of Color: chocolate.


## 3. Runs: BM25, dense, hybrid on each text version

In [3]:
from rank_bm25 import BM25Okapi

tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
runs = {}
for version, text in TEXTS.items():
    bm25 = BM25Okapi([tok(t) for t in text])
    runs["bm25", version] = {q: list(pids[np.argsort(-bm25.get_scores(tok(qtext[q])))[:100]]) for q in all_q}
    print("bm25", version, "done")

bm25 base done
bm25 +color done


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
CACHE = {"base": "wands_emb.npy", "+color": "wands_emb_color.npy"}   # base = same cache as notebooks 03, 04, eval
Qe = model.encode([qtext[q] for q in all_q], normalize_embeddings=True)

for version, text in TEXTS.items():
    if os.path.exists(CACHE[version]):
        E = np.load(CACHE[version])
    else:
        E = model.encode(text.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        np.save(CACHE[version], E)
    S = Qe @ E.T
    runs["dense", version] = {q: list(pids[np.argsort(-S[i])[:100]]) for i, q in enumerate(all_q)}
    runs["hybrid_rrf", version] = {q: rrf([runs["bm25", version][q], runs["dense", version][q]]) for q in all_q}
    print("dense + hybrid", version, "done")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dense + hybrid base done
dense + hybrid +color done


## 4. Part A results: does adding the color to the text change behaviour?

In [5]:
K = 10
LENIENT = True
color_qids = [q for q in all_q if query_colors(qtext[q], LENIENT)]
print("queries naming a color:", len(color_qids))

def grader(qid):
    qcols = query_colors(qtext[qid], LENIENT)
    return lambda p: color_check(qcols, pcolor.get(p, ""), pname.get(p, ""), LENIENT)

def ndcg_per_query(run, qids):
    return [ndcg_at_k(run[q], qrels.get(q, {}), K) for q in qids]

def viol_per_query(run, qids, among="checkable"):
    return [constraint_violation_rate(run[q], grader(q), K, among) for q in qids]

SYSTEMS = ["bm25", "dense", "hybrid_rrf"]
relevance = pd.DataFrame({
    (s, v): {"NDCG@10 (all 480)": np.mean(ndcg_per_query(runs[s, v], all_q)),
             "NDCG@10 (color queries)": np.mean(ndcg_per_query(runs[s, v], color_qids))}
    for s in SYSTEMS for v in TEXTS}).T
constraint = pd.DataFrame({
    (s, v): {"violation_rate (checkable)": np.nanmean(viol_per_query(runs[s, v], color_qids)),
             "violating share of top-10": np.mean(viol_per_query(runs[s, v], color_qids, "k"))}
    for s in SYSTEMS for v in TEXTS}).T
print("Relevance"); display(relevance.round(3))
print("Constraint"); display(constraint.round(3))

queries naming a color: 44
Relevance


NDCG@10 (all 480)  NDCG@10 (color queries)
bm25       base                0.687                    0.533
           +color              0.687                    0.533
dense      base                0.697                    0.560
           +color              0.698                    0.554
hybrid_rrf base                0.748                    0.644
           +color              0.747                    0.625

Constraint


violation_rate (checkable)  violating share of top-10
bm25       base                         0.316                      0.195
           +color                       0.208                      0.114
dense      base                         0.592                      0.323
           +color                       0.453                      0.261
hybrid_rrf base                         0.416                      0.220
           +color                       0.282                      0.143

In [6]:
print("Relevance: NDCG@10 difference, +color minus base (color queries / all queries)")
for s in SYSTEMS:
    for name, qids in [("color", color_qids), ("all", all_q)]:
        m, (lo, hi), n = paired_bootstrap(ndcg_per_query(runs[s, "+color"], qids), ndcg_per_query(runs[s, "base"], qids))
        print(f"  {s:10s} {name:5s}: {m:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  (n={n})")

print("\nConstraint: violation-rate difference, +color minus base (color queries)")
for s in SYSTEMS:
    m, (lo, hi), n = paired_bootstrap(viol_per_query(runs[s, "+color"], color_qids), viol_per_query(runs[s, "base"], color_qids))
    print(f"  {s:10s}: {m:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  (n={n})")

print("\nDoes the dense - BM25 violation gap survive once both see the color?")
for v in TEXTS:
    m, (lo, hi), n = paired_bootstrap(viol_per_query(runs["dense", v], color_qids), viol_per_query(runs["bm25", v], color_qids))
    print(f"  {v:7s}: dense - bm25 = {m:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  (n={n})")

Relevance: NDCG@10 difference, +color minus base (color queries / all queries)
  bm25       color: +0.000  95% CI [-0.025, +0.023]  (n=44)
  bm25       all  : -0.000  95% CI [-0.004, +0.003]  (n=480)
  dense      color: -0.005  95% CI [-0.024, +0.012]  (n=44)
  dense      all  : +0.001  95% CI [-0.005, +0.006]  (n=480)
  hybrid_rrf color: -0.020  95% CI [-0.052, +0.009]  (n=44)
  hybrid_rrf all  : -0.001  95% CI [-0.006, +0.004]  (n=480)

Constraint: violation-rate difference, +color minus base (color queries)
  bm25      : -0.108  95% CI [-0.178, -0.041]  (n=40)
  dense     : -0.154  95% CI [-0.204, -0.106]  (n=36)
  hybrid_rrf: -0.134  95% CI [-0.201, -0.069]  (n=38)

Does the dense - BM25 violation gap survive once both see the color?
  base   : dense - bm25 = +0.259  95% CI [+0.180, +0.342]  (n=36)
  +color : dense - bm25 = +0.212  95% CI [+0.102, +0.311]  (n=36)


## 5. Part B: where does the hypothesis fail?

Per color query, on the **base** text: dense − BM25 violation rate (checkable). *Worse* = dense violates more, *not worse* = equal or
less, *undefined* = one of the two has nothing checkable in its top 10.

In [7]:
def checkable_share(run, q):
    g = grader(q)
    return np.mean([g(p) in ("match", "mismatch") for p in run[q][:K]])

rows = []
for q in color_qids:
    d = viol_per_query(runs["dense", "base"], [q])[0]
    b = viol_per_query(runs["bm25", "base"], [q])[0]
    rows.append({"qid": q, "query": qtext[q], "query_class": qclass[q],
                 "colors": ", ".join(sorted(query_colors(qtext[q], LENIENT))),
                 "words": len(qtext[q].split()),
                 "bm25_viol": b, "dense_viol": d, "diff": d - b,
                 "dense_checkable": checkable_share(runs["dense", "base"], q)})
per_q = pd.DataFrame(rows)
per_q["outcome"] = np.select([per_q["diff"].isna(), per_q["diff"] > 0], ["undefined", "dense worse"], "dense not worse")
print(per_q["outcome"].value_counts().to_string())

outcome
dense worse        29
undefined           8
dense not worse     7


In [8]:
print("Queries where the hypothesis FAILS (dense not worse than BM25):")
display(per_q[per_q["outcome"] == "dense not worse"]
        .sort_values("diff")[["query", "query_class", "colors", "bm25_viol", "dense_viol", "dense_checkable"]].round(2))

Queries where the hypothesis FAILS (dense not worse than BM25):


,query,query_class,colors,bm25_viol,dense_viol,dense_checkable
26,white splashproof shiplap wallpaper,Wallpaper,white,0.57,0.33,0.6
20,blue faux throw with pillows,Accent Pillows,blue,0.60,0.50,1.0
18,black frame 22 x 36,Picture Frames,black,0.29,0.22,0.9
42,small wardrobe grey,Armoires & Wardrobes,gray,0.33,0.30,1.0
9,tollette teal outdoor rug,Area Rugs,teal,0.20,0.20,1.0
10,royal blue counter height chairs,Bar Stools,blue,0.67,0.67,0.3
31,kohler purist brushed bronze,Bathroom Sink Faucets,bronze,1.00,1.00,0.1


In [9]:
print("What differs between the two groups (means):")
display(per_q[per_q["outcome"] != "undefined"].groupby("outcome")[["words", "bm25_viol", "dense_viol", "dense_checkable"]].mean().round(2))

print("By query color (count of queries per outcome):")
display(pd.crosstab(per_q["colors"], per_q["outcome"]))

What differs between the two groups (means):


,words,bm25_viol,dense_viol,dense_checkable
outcome,,,,
dense not worse,4.29,0.52,0.46,0.70
dense worse,4.21,0.29,0.62,0.72


By query color (count of queries per outcome):


outcome,dense not worse,dense worse,undefined
colors,,,
"beige, black",0,1,0
black,1,5,1
"black, white",0,0,1
blue,2,3,1
bronze,1,2,1
gold,0,3,0
gray,1,3,1
green,0,0,1
orange,0,1,0


**How to read Part B.** The hypothesis is about averages. These tables show the queries where it does not hold and whether
they share something (a color, a product class, short queries, products with little color data). Use them in the report's
"where the hypothesis fails" paragraph, with two or three concrete example queries.

## 6. Is it specific to MiniLM? A second, larger dense model

`all-mpnet-base-v2` is larger (768 dimensions vs 384) and generally stronger. If it also violates color far more than BM25, the finding
is about dense retrieval, not one model. Base text only.

**Time:** encoding the 42,994 products with this model takes noticeably longer than MiniLM on a CPU (cached after the first run).

In [10]:
MODEL2 = "sentence-transformers/all-mpnet-base-v2"
model2 = SentenceTransformer(MODEL2)
CACHE2 = "wands_emb_mpnet.npy"
if os.path.exists(CACHE2):
    E2 = np.load(CACHE2)
else:
    E2 = model2.encode(TEXTS["base"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(CACHE2, E2)

Qe2 = model2.encode([qtext[q] for q in all_q], normalize_embeddings=True)
S2 = Qe2 @ E2.T
runs["dense_mpnet", "base"] = {q: list(pids[np.argsort(-S2[i])[:100]]) for i, q in enumerate(all_q)}
runs["hybrid_mpnet", "base"] = {q: rrf([runs["bm25", "base"][q], runs["dense_mpnet", "base"][q]]) for q in all_q}
print("mpnet done")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/672 [00:00<?, ?it/s]

mpnet done


In [11]:
SYS2 = ["bm25", "dense", "dense_mpnet", "hybrid_rrf", "hybrid_mpnet"]
rel2 = pd.DataFrame({s: {"NDCG@10 (all 480)": np.mean(ndcg_per_query(runs[s, "base"], all_q)),
                         "NDCG@10 (color queries)": np.mean(ndcg_per_query(runs[s, "base"], color_qids))} for s in SYS2}).T
con2 = pd.DataFrame({s: {"violation_rate (checkable)": np.nanmean(viol_per_query(runs[s, "base"], color_qids)),
                         "violating share of top-10": np.mean(viol_per_query(runs[s, "base"], color_qids, "k"))} for s in SYS2}).T
print("Relevance"); display(rel2.round(3))
print("Constraint"); display(con2.round(3))

for a, b in [("dense_mpnet", "bm25"), ("dense_mpnet", "dense")]:
    m1, (l1, h1), n1 = paired_bootstrap(ndcg_per_query(runs[a, "base"], color_qids), ndcg_per_query(runs[b, "base"], color_qids))
    m2, (l2, h2), n2 = paired_bootstrap(viol_per_query(runs[a, "base"], color_qids), viol_per_query(runs[b, "base"], color_qids))
    print(f"{a} - {b}:  NDCG {m1:+.3f} [{l1:+.3f}, {h1:+.3f}] (n={n1})   violation {m2:+.3f} [{l2:+.3f}, {h2:+.3f}] (n={n2})")

Relevance


,NDCG@10 (all 480),NDCG@10 (color queries)
bm25,0.687,0.533
dense,0.697,0.560
dense_mpnet,0.684,0.563
hybrid_rrf,0.748,0.644
hybrid_mpnet,0.752,0.670


Constraint


,violation_rate (checkable),violating share of top-10
bm25,0.316,0.195
dense,0.592,0.323
dense_mpnet,0.528,0.330
hybrid_rrf,0.416,0.220
hybrid_mpnet,0.390,0.234


dense_mpnet - bm25:  NDCG +0.030 [-0.046, +0.112] (n=44)   violation +0.192 [+0.101, +0.284] (n=39)
dense_mpnet - dense:  NDCG +0.003 [-0.049, +0.059] (n=44)   violation -0.084 [-0.166, -0.008] (n=36)
